# GOZ-Code-Extraktion: Training & Inferenz (Colab)

Dieses Notebook ist für **Google Colab mit T4-GPU** gebaut (QLoRA-Training
braucht eine GPU, die es in der lokalen Entwicklungsumgebung dieses Projekts
nicht gibt). Es deckt zwei Ansätze ab, die später gegeneinander evaluiert
werden (siehe Task 11, Eval-Report):

1. **RAG-Baseline**: Basismodell (`meta-llama/Llama-3.2-3B-Instruct`) +
   BM25/Embedding-Retrieval über die GOZ-Codeliste als Prompt-Kontext.
2. **QLoRA-Finetune**: dasselbe Basismodell, per LoRA auf den generierten
   Trainingsdaten (`data/train.jsonl`) feingetunt, ohne Retrieval-Kontext
   zur Inferenzzeit (Wissen steckt in den LoRA-Gewichten).

Beide Ansätze laufen am Ende über dasselbe Test-Set (`data/test.jsonl`)
und schreiben ihre Vorhersagen nach `results/predictions_rag.jsonl` bzw.
`results/predictions_finetune.jsonl` — das gemeinsame Format pro Zeile ist
`{"text": ..., "expected_codes": [...], "predicted_codes": [...]}`.

**Wichtig:** Für `meta-llama/Llama-3.2-3B-Instruct` muss die Lizenz auf
Hugging Face vorher akzeptiert und ein HF-Token mit Zugriff bereitliegen
(siehe Zelle 2).


## 1. Setup & Daten laden

Installiert alle Abhängigkeiten, loggt sich bei Hugging Face ein (nötig für
den Download des lizenzpflichtigen Llama-3.2-Modells), lädt die Projekt-
Dateien hoch und liest die kuratierten GOZ-Codes sowie Train-/Test-Split.


In [ ]:
!pip install -q transformers peft trl bitsandbytes accelerate datasets sentence-transformers rank-bm25 pydantic


In [ ]:
from huggingface_hub import notebook_login
notebook_login()  # HF-Token mit akzeptierter Llama-3.2-Lizenz eingeben


In [ ]:
# Repo-Dateien hochladen: src/goz_extract/, data/goz_codes.json, data/train.jsonl, data/test.jsonl
from google.colab import files
uploaded = files.upload()  # als .zip hochladen und entpacken, siehe README-Setup-Abschnitt
!unzip -o goz-extract-src.zip -d .
import sys; sys.path.insert(0, "src")


In [ ]:
import json
from pathlib import Path

from goz_extract.schema import GozCode, NoteExample

codes = [GozCode(**c) for c in json.loads(Path("data/goz_codes.json").read_text(encoding="utf-8"))]
train = [NoteExample.model_validate_json(l) for l in Path("data/train.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
test = [NoteExample.model_validate_json(l) for l in Path("data/test.jsonl").read_text(encoding="utf-8").splitlines() if l.strip()]
valid_codes = {c.goz_nr for c in codes}
print(len(codes), len(train), len(test))


**Erwartete Ausgabe:** `55 <train-Anzahl> <test-Anzahl>` (Zahlen aus Task 5, Step 6).


## 2. RAG-Baseline

Baut BM25- und Embedding-Index über die GOZ-Codeliste auf (kombiniert per
Reciprocal Rank Fusion in `retrieve_candidates`, siehe Task 6), lädt das
unveränderte Basismodell und lässt es pro Test-Notiz mit den Top-12-
Retrieval-Kandidaten als Prompt-Kontext antworten.


In [ ]:
from sentence_transformers import SentenceTransformer

from goz_extract.retrieval import BM25Index, EmbeddingIndex, retrieve_candidates

embed_model = SentenceTransformer("intfloat/multilingual-e5-base")


def encode_fn(texts):
    return embed_model.encode([f"passage: {t}" for t in texts], normalize_embeddings=False)


bm25_index = BM25Index(codes)
embedding_index = EmbeddingIndex(codes, encode_fn=encode_fn)
code_by_nr = {c.goz_nr: c for c in codes}


In [ ]:
from goz_extract.inference import generate_codes, load_model

base_model, tokenizer = load_model("meta-llama/Llama-3.2-3B-Instruct")

rag_predictions = []
for example in test:
    candidate_codes = [code_by_nr[nr] for nr in retrieve_candidates(example.text, bm25_index, embedding_index, top_n=12)]
    predicted = generate_codes(base_model, tokenizer, example.text, valid_codes, candidates=candidate_codes)
    rag_predictions.append({"text": example.text, "expected_codes": example.expected_codes, "predicted_codes": predicted})

print(rag_predictions[0])


**Erwartetes Verhalten:** läuft ohne Fehler über alle Test-Notizen durch
(Dauer: ca. 1–3 Minuten auf T4 für ~80–100 Notizen), `predicted_codes` in
jedem Eintrag ist eine Liste gültiger GOZ-Ziffern.


In [ ]:
import json
Path("results").mkdir(exist_ok=True)
with open("results/predictions_rag.jsonl", "w", encoding="utf-8") as f:
    for row in rag_predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")


## 3. LoRA-Training (QLoRA)

Baut aus den Trainingsbeispielen Chat-formatierte Texte (Prompt ohne
Retrieval-Kandidaten — das Finetune soll die Codes aus den Gewichten lernen,
nicht aus Kontext), lädt das Basismodell zusätzlich 4-bit-quantisiert für
QLoRA und trainiert einen LoRA-Adapter darauf.


In [ ]:
from datasets import Dataset
from goz_extract.prompting import build_extraction_prompt

def to_chat_example(example):
    prompt = build_extraction_prompt(example.text, candidates=None)
    completion = ", ".join(example.expected_codes)
    messages = [{"role": "user", "content": prompt}, {"role": "assistant", "content": completion}]
    return {"messages": messages}

train_dataset = Dataset.from_list([to_chat_example(e) for e in train])


In [ ]:
import torch
from peft import LoraConfig
from transformers import BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

from transformers import AutoModelForCausalLM

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_compute_dtype=torch.float16
)
# Fürs Training brauchen wir eine 4-bit-quantisierte Kopie (QLoRA) - nicht
# über load_model() (das lädt in bfloat16 ohne Quantisierung, siehe Task 9),
# sondern direkt mit quantization_config.
train_model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.2-3B-Instruct", quantization_config=bnb_config, device_map="auto"
)

lora_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

# assistant_only_loss=True: Loss nur auf den Assistant-Antworten, nicht auf
# der (bei fast allen Beispielen identischen) Instruktion+Notiz davor. Ohne
# das lernt SFTTrainer per Default über die komplette Sequenz - führte live
# zu Mode Collapse (73% identische Vorhersage unabhängig von der Notiz),
# weil das Lernsignal für die eigentliche Code-Auswahl im sich
# wiederholenden Prompt-Teil unterging. Braucht ein "messages"-Feld im
# Dataset (siehe vorherige Zelle) statt eines geflachten "text"-Strings -
# in TRL >=1.x ersetzt das den älteren DataCollatorForCompletionOnlyLM.
sft_config = SFTConfig(
    output_dir="adapters/goz-extract-llama32-3b",
    num_train_epochs=3,
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    logging_steps=10,
    save_strategy="epoch",
    assistant_only_loss=True,
)

trainer = SFTTrainer(
    model=train_model,
    train_dataset=train_dataset,
    args=sft_config,
    peft_config=lora_config,
)
trainer.train()
trainer.save_model("adapters/goz-extract-llama32-3b")


**Erwartetes Verhalten:** Trainings-Loss sinkt sichtbar über die
Logging-Schritte; läuft ohne CUDA-OOM auf T4 durch (bei OOM:
`per_device_train_batch_size` auf 2 reduzieren, `gradient_accumulation_steps`
auf 8 erhöhen).

> Hinweis: `import torch` wurde am Anfang dieser Zelle ergänzt (im
> Task-Brief fehlte der Import, `torch.bfloat16` wird aber direkt darunter
> gebraucht) — sonst ist die Zelle unverändert wie im Brief spezifiziert.


## 4. Finetune-Inferenz

Lädt das Basismodell zusammen mit dem gerade trainierten LoRA-Adapter und
lässt es — diesmal ohne Retrieval-Kandidaten im Prompt — über dieselben
Test-Notizen wie die RAG-Baseline laufen, damit beide Ansätze direkt
vergleichbar sind (Task 11).


Bevor das feingetunte Modell geladen wird: `base_model` (RAG-Baseline,
bf16/fp16), `train_model` (4-bit-QLoRA-Trainingskopie) und `trainer`
liegen zu diesem Zeitpunkt noch im GPU-Speicher. Zusammen mit einer
dritten Modellkopie für die Finetune-Inferenz würde das auf einer T4
(~15GB VRAM) sehr wahrscheinlich zu einem CUDA-Out-of-Memory führen.
Die folgende Zelle gibt den Speicher der nicht mehr gebrauchten
Objekte frei, bevor `finetuned_model` geladen wird.


In [ ]:
import gc, torch

del base_model, train_model, trainer
gc.collect()
torch.cuda.empty_cache()


In [ ]:
finetuned_model, ft_tokenizer = load_model(
    "meta-llama/Llama-3.2-3B-Instruct", adapter_path="adapters/goz-extract-llama32-3b"
)

finetune_predictions = []
for example in test:
    predicted = generate_codes(finetuned_model, ft_tokenizer, example.text, valid_codes, candidates=None)
    finetune_predictions.append({"text": example.text, "expected_codes": example.expected_codes, "predicted_codes": predicted})

with open("results/predictions_finetune.jsonl", "w", encoding="utf-8") as f:
    for row in finetune_predictions:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print(finetune_predictions[0])


**Erwartetes Verhalten:** läuft über alle Test-Notizen durch,
`predicted_codes` enthält gültige GOZ-Ziffern.


## 5. Artefakte herunterladen & committen (manueller Schritt)

Die folgenden zwei Schritte passieren **außerhalb** dieses Notebooks, von
einem Menschen, nachdem die Zellen oben auf Colab erfolgreich durchgelaufen
sind — sie sind hier nur dokumentiert, nicht als ausführbare Zellen, weil
sie lokalen Datei-/Git-Zugriff brauchen, den Colab nicht hat:

1. **Artefakte herunterladen:** `results/predictions_rag.jsonl`,
   `results/predictions_finetune.jsonl` und den Ordner
   `adapters/goz-extract-llama32-3b/` aus Colab herunterladen
   (Dateibrowser oder `files.download(...)`) und lokal nach
   `goz-finetune-vs-rag/results/` bzw. `goz-finetune-vs-rag/adapters/`
   legen.
2. **Committen:**
   ```bash
   git add notebooks/train_and_infer.ipynb results/predictions_rag.jsonl results/predictions_finetune.jsonl
   git commit -m "Add Colab notebook for QLoRA training and RAG-baseline/finetune inference over the test set"
   ```
   (LoRA-Adapter-Gewichte unter `adapters/` sind groß — vor dem Commit
   prüfen, ob sie stattdessen per `.gitignore` ausgeschlossen und separat
   z. B. auf Hugging Face Hub hochgeladen werden sollen; im README
   verlinken.)
